In [1]:
import torch
import pandas as pd
import numpy as np
import random
from pyfaidx import Fasta

import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

from akita.model import SeqNN

In [ ]:
from utils.data_utils import (one_hot_encode_sequence, 
                              upper_triangular_to_vector,
                              fragment_indices_in_upper_triangular)

In [2]:
# this could be moved to utils as well

def store_tower_output(ohe_sequence, model, path):
    x = model.conv_block_1(ohe_sequence)
    x = model.conv_tower(x)
    # save the tensor
    torch.save(x, path)
    torch.cuda.empty_cache()

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
model_idx = 0

In [5]:
model = SeqNN()
model.load_state_dict(torch.load(f"/home1/smaruj/pytorch_akita/models/finetuned/mouse/Hsieh2019_mESC/checkpoints/Akita_v2_mouse_Hsieh2019_mESC_model{model_idx}_finetuned.pth", map_location=device))
model.eval()

SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 128, kernel_size=(15,), stride=(1,), padding=(7,), bias=False)
    (batch_norm): BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size=2, stride=2, paddi

In [6]:
FOLD = 7

In [7]:
df = pd.read_csv(f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/analysis/flat_regions/mouse_flat_regions_chrom_states_tsv/fold{FOLD}_selected_genomic_windows_centered_chrom_states.tsv", sep="\t")

In [8]:
fasta_file = "/project2/fudenber_735/genomes/mm10/mm10.fa"
genome = Fasta(fasta_file)

In [9]:
c = -0.5 # how strong target boundary is

In [10]:
size = 512
mask = np.zeros((size, size))
half = size // 2

# creating a boundary
mask[:half, half:] = c
mask[half:, :half] = c

In [11]:
boundary_mask_vector = upper_triangular_to_vector(mask, 2)
boundary_mask_vector_tensor = torch.tensor(boundary_mask_vector).to(device)

In [12]:
# Create a blank mask
fragment_bool_mask = np.zeros((size, size), dtype=bool)

# Mark the specified fragment
fragment_bool_mask[:half, half:] = True
fragment_bool_mask[half:, :half] = True

In [13]:
boundary_mask_indices = fragment_indices_in_upper_triangular(matrix_size=size, fragment_mask=fragment_bool_mask)
boundary_mask_indices_tensor = torch.tensor(boundary_mask_indices)

In [ ]:
torch.save(boundary_mask_indices_tensor, "path-to-store-target-masks/boundary_indices.pt")

In [14]:
for row in df.itertuples(index=False):
    chrom, pred_start, pred_end = row.chrom, row.centered_start, row.centered_end
    sequence = genome[row.chrom][pred_start:pred_end]
    
    X = one_hot_encode_sequence(sequence)
    X_tensor = torch.tensor(X)
    torch.save(X_tensor, f"path-to-store-genome-flat-sequences/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_X.pt")

    store_tower_output(X_tensor, model, f"path-to-store-their-tower-outputs/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_tower_out.pt")
    
    model.eval()
    with torch.no_grad():
        y = model(X_tensor)
        
    # replacing values of map with the mask
    y_bar = y.to(device).clone()
    masked_values = boundary_mask_vector_tensor[boundary_mask_indices_tensor].float()
    y_bar[0, 0, boundary_mask_indices_tensor] = masked_values.to(device)
    
    torch.save(y_bar, f"path-to-store-boundary-target/target_{c}/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_target.pt")
    